In [4]:
!pip install 'transformers<5.0.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.3 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [1]:
import transformers
transformers.__version__

'4.57.6'

In [4]:
from transformers import TFBertForMaskedLM, BertTokenizer
import tensorflow as tf

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# Forcing use_safetensors=False to avoid the safe_open library error under Python 3.13
model = TFBertForMaskedLM.from_pretrained('bert-base-uncased', use_safetensors=False)

# Input sentence with proper mask token
input_text = f"The cat sat on the {tokenizer.mask_token} and looked out the window."

# Tokenize
inputs = tokenizer(input_text, return_tensors="tf")

# Run model
outputs = model(inputs)
logits = outputs.logits

# Find index of [MASK] token (using correct indexing for TF)
mask_token_index = tf.where(inputs["input_ids"] == tokenizer.mask_token_id)[0, 1].numpy()

# Predict token at [MASK]
predicted_token_id = tf.argmax(logits[0, mask_token_index]).numpy()
predicted_token = tokenizer.decode([predicted_token_id])

top_k = 5
top_k_ids = tf.math.top_k(logits[0, mask_token_index], k=top_k).indices.numpy()
top_k_tokens = tokenizer.convert_ids_to_tokens(top_k_ids)

print("Top predictions:")
for i, token in enumerate(top_k_tokens, 1):
    print(f"{i}: {token}")

tf_model.h5:   0%|          | 0.00/536M [00:00<?, ?B/s]

All model checkpoint layers were used when initializing TFBertForMaskedLM.

All the layers of TFBertForMaskedLM were initialized from the model checkpoint at bert-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForMaskedLM for predictions without further training.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Top predictions:
1: floor
2: bed
3: couch
4: sofa
5: ground


In [6]:
!pip install -q tensorflow transformers datasets

import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification, create_optimizer
from datasets import load_dataset

# 1. Load small IMDB subset
dataset = load_dataset("imdb")
small_train = dataset["train"].select(range(500))
small_test = dataset["test"].select(range(100))

# 2. Load tokenizer & model
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
# Added use_safetensors=False to fix Python 3.13 'safe_open' TypeError
model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", use_safetensors=False)

# 3. Tokenize the text
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

small_train = small_train.map(tokenize, batched=True)
small_test = small_test.map(tokenize, batched=True)

small_train.set_format(type="tensorflow", columns=["input_ids", "attention_mask", "label"])
small_test.set_format(type="tensorflow", columns=["input_ids", "attention_mask", "label"])

train_tf_dataset = small_train.to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols="label",
    shuffle=True,
    batch_size=16
)

test_tf_dataset = small_test.to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols="label",
    shuffle=False,
    batch_size=16
)

# 4. Hugging Face create_optimizer (FIX)
num_train_steps = len(train_tf_dataset) * 1  # epochs = 1
optimizer, lr_schedule = create_optimizer(
    init_lr=5e-5,
    num_train_steps=num_train_steps,
    num_warmup_steps=0
)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# 5. Train model (1 epoch)
model.fit(train_tf_dataset, validation_data=test_tf_dataset, epochs=1)

# 6. Predict on custom input
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="tf", truncation=True, padding="max_length", max_length=128)
    logits = model(inputs).logits
    probs = tf.nn.softmax(logits, axis=1).numpy()[0]
    label = "Positive ​😊​" if probs[1] > probs[0] else "Negative ​😞​"
    confidence = round(probs.max(), 2)
    return label, confidence

# 7. Example predictions
examples = [
    "This movie was amazing! I loved every moment.",
    "Worst movie ever. Total waste of time.",
    "It was okay, not great but not terrible either."
]

for example in examples:
    sentiment, conf = predict_sentiment(example)
    print(f"\nInput: {example}\nPrediction: {sentiment} (Confidence: {conf})")

tf_model.h5:   0%|          | 0.00/363M [00:00<?, ?B/s]

Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['vocab_projector', 'vocab_transform', 'vocab_layer_norm', 'activation_13']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier', 'classifier', 'dropout_150']
You should probably TRAIN this model on a down-stream task to be able to use 

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

32/32 [==============================] - 549s 16s/step - loss: 0.1345 - accuracy: 0.9700 - val_loss: 0.0178 - val_accuracy: 1.0000

Input: This movie was amazing! I loved every moment.
Prediction: Negative ​😞​ (Confidence: 0.9700000286102295)

Input: Worst movie ever. Total waste of time.
Prediction: Negative ​😞​ (Confidence: 0.9700000286102295)

Input: It was okay, not great but not terrible either.
Prediction: Negative ​😞​ (Confidence: 0.9700000286102295)
